In [1]:
import torch
from PIL import Image
from transformers import XLMRobertaTokenizer
from torchvision import transforms

# BEiT-3 레포지토리의 모델링 파일을 import해야 합니다.
# 이 파일이 있는 경로를 sys.path에 추가하거나, 같은 디렉토리에 복사해두세요.
from modeling_finetune import beit3_large_patch16_768_vqav2

import json # VQA 답변 라벨을 로드하기 위해 import

/data/2_data_server/cv-07/anaconda3/envs/beit3_env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- 1. 설정 (사용자 환경에 맞게 수정) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fine-tuning된 모델 가중치 파일 경로
model_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/beit3_large_indomain_patch16_768_vgqaaug_vqa.pth"
# 문장 토크나이저 모델 경로
spm_model_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/beit3.spm"
# VQAv2 답변 라벨 파일 경로 (데이터 준비 과정에서 생성됨)
# 실제 VQAv2 데이터셋의 answer list가 필요합니다.
# 예시: https://github.com/microsoft/unilm/blob/master/beit3/get_started/get_started_for_vqav2.md
vqa_answer_label_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/v2_mscoco_train2014_annotations.json"

# --- 모델 및 토크나이저 불러오기 ---
tokenizer = XLMRobertaTokenizer(spm_model_path)

In [3]:
# 2. import한 함수를 바로 호출하여 모델 객체를 생성합니다.
#    이 함수가 내부적으로 BEiT3ForVisualQuestionAnswering 클래스를 사용해 모델을 만듭니다.
print("Creating model...")
model = beit3_large_patch16_768_vqav2()

Creating model...


In [4]:
# 3. Fine-tuning된 가중치를 불러옵니다. (이 부분은 동일)
print("Loading checkpoint...")
checkpoint = torch.load(model_path, map_location='cpu')
model.load_state_dict(checkpoint['model'])

model.to(device)
model.eval() # 모델을 평가 모드로 설정

Loading checkpoint...


BEiT3ForVisualQuestionAnswering(
  (beit3): BEiT3(
    (text_embed): TextEmbedding(64010, 1024)
    (vision_embed): VisionEmbedding(
      (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
    )
    (encoder): Encoder(
      (dropout_module): Dropout(p=0.0, inplace=False)
      (embed_positions): MutliwayEmbedding(
        (A): PositionalEmbedding(2307, 1024)
        (B): PositionalEmbedding(1024, 1024)
      )
      (layers): ModuleList(
        (0): EncoderLayer(
          (self_attn): MultiheadAttention(
            (k_proj): MultiwayNetwork(
              (A): Linear(in_features=1024, out_features=1024, bias=True)
              (B): Linear(in_features=1024, out_features=1024, bias=True)
            )
            (v_proj): MultiwayNetwork(
              (A): Linear(in_features=1024, out_features=1024, bias=True)
              (B): Linear(in_features=1024, out_features=1024, bias=True)
            )
            (q_proj): MultiwayNetwork(
              (A): Linear(in_feat

In [5]:
from collections import Counter

print(f"Loading raw annotation file from: {vqa_answer_label_path}")
with open(vqa_answer_label_path, 'r') as f:
    vqa_data = json.load(f)

# 'annotations' 키에 해당하는 리스트를 가져옵니다.
raw_annotations = vqa_data['annotations']
print(f"Found {len(raw_annotations)} annotations.")

# --- 2. 모든 답변들의 빈도를 계산합니다. ---
# Counter는 각 항목의 개수를 세어주는 유용한 도구입니다.
answer_counter = Counter()
for ann in raw_annotations:
    # VQAv2에서는 여러 답변 중 'multiple_choice_answer'가 최종 정답으로 간주됩니다.
    answer_counter[ann['multiple_choice_answer']] += 1

# --- 3. 가장 빈도수가 높은 3,129개의 답변을 선택합니다. ---
# 이것이 VQAv2의 표준 답변 후보 개수입니다.
num_top_answers = 3129
top_answers = answer_counter.most_common(num_top_answers)
answer_vocab = [answer for answer, count in top_answers]
print(f"Top {len(answer_vocab)} answers extracted.")

# --- 4. 최종적으로 사용할 answer_map (인덱스 -> 답변)을 생성합니다. ---
answer_map = {str(i): answer for i, answer in enumerate(answer_vocab)}
print("Successfully created the 'answer_map' dictionary!")

# --- 5. 생성된 answer_map 내용 일부 확인 (선택 사항) ---
print("\n--- Example of created answer_map ---")
for i in range(10):
    print(f"Index {i}: {answer_map[str(i)]}")

Loading raw annotation file from: /data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/v2_mscoco_train2014_annotations.json
Found 443757 annotations.
Top 3129 answers extracted.
Successfully created the 'answer_map' dictionary!

--- Example of created answer_map ---
Index 0: yes
Index 1: no
Index 2: 1
Index 3: 2
Index 4: white
Index 5: 3
Index 6: blue
Index 7: red
Index 8: black
Index 9: 0


In [6]:
transform = transforms.Compose([
    transforms.Resize((768, 768)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])
image = Image.open(image_path).convert("RGB")
image_tensor = transform(image).unsqueeze(0).to(device)

NameError: name 'image_path' is not defined

In [ ]:
count = 0

import torch
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
import pandas as pd
import os
import re

# --- 1. 기본 설정 및 데이터 로드 ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_csv_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv'
image_dir = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/'
test = pd.read_csv(test_csv_path)
results = []

# --- 2. 이미지 전처리(Transform) 정의 ---
transform = transforms.Compose([
    transforms.Resize((768, 768)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

# --- 3. 답변 어휘 사전 준비 (★★ 핵심 준비 단계 ★★) ---
# answer_map (인덱스 -> 답변)은 이전 셀에서 생성되었다고 가정합니다.
# "답변 -> 인덱스" 역방향 맵을 생성합니다.
# 이 맵이 있어야 A,B,C,D 선택지의 점수를 효율적으로 찾을 수 있습니다.
try:
    answer_vocab = list(answer_map.values())
    answer_to_idx = {answer: i for i, answer in enumerate(answer_vocab)}
    print("Answer to Index mapping created successfully.")
except NameError:
    print("Error: 'answer_map' is not defined. Please create it first.")
    # 스크립트 중단 또는 기본값 설정
    answer_to_idx = {}


# --- 4. 루프를 돌며 추론 실행 ---
print("Starting inference with choice scoring...")
for _, row in tqdm(test.iterrows(), total=len(test)):
    # 4-1. 이미지 불러오기 및 전처리
    image_path = os.path.join(image_dir, row['img_path'].lstrip('./'))
    if not os.path.exists(image_path):
        results.append({'img_path': row['img_path'], 'predicted_answer': '?'})
        continue
    
    try:
        image = Image.open(image_path).convert("RGB")
        image_tensor = transform(image).unsqueeze(0).to(device)
    except Exception as e:
        print(f"Error processing image {image_path}: {e}")
        results.append({'img_path': row['img_path'], 'predicted_answer': '?'})
        continue

    # 4-2. 질문 텍스트 전처리 (순수한 질문만 사용)
    question_text = row['Question']
    encoding = tokenizer(
        question_text,
        padding='max_length',
        truncation=True,
        max_length=40,
        return_tensors='pt'
    )
    question_tokens = encoding['input_ids'].to(device)
    padding_mask = encoding['attention_mask'].to(device)

    # 4-3. 모델 추론 실행 (전체 답변에 대한 점수 획득)
    with torch.no_grad():
        output_logits = model(image=image_tensor, question=question_tokens, padding_mask=padding_mask)

    # 4-4. 각 보기의 점수를 비교하여 최적의 답변 선택 (★★ 핵심 로직 ★★)
    choices = [row[c] for c in ['A', 'B', 'C', 'D']]
    best_score = -float('inf')

    # 기본값은 첫번째 캡션
    predicted_answer = choices[0]  # 기본값은 첫번째 보기로 설정

    for choice in choices:
        # 현재 보기가 모델의 답변 어휘 사전에 있는지 확인
        # 소문자로 변환하여 비교하면 더 정확할 수 있습니다. (e.g., "Cat" vs "cat")
        choice_lower = choice.lower()
        if choice_lower in answer_to_idx:
            # 해당 보기의 인덱스를 가져옴
            idx = answer_to_idx[choice_lower]
            # 전체 점수(logits)에서 해당 인덱스의 점수를 가져옴
            score = output_logits[0, idx].item()
            
            # 현재 점수가 최고 점수보다 높으면, 최고 점수와 정답을 업데이트
            if score > best_score:
                best_score = score
                predicted_answer = choice
        # else:
            # 만약 보기가 모델의 사전에 없다면 처리하지 않음
            # print(f"Choice '{choice}' not in answer vocabulary.")

    # 4-5. 최종 결과 저장
    results.append({
        'img_path': row['img_path'],
        'question': row['Question'],
        'predicted_answer': predicted_answer
    })

    # if count < 10:
    #     print(f"Image: {row['img_path']}, Question: {row['Question']}, Predicted Answer: {predicted_answer}")
    #     count += 1
    # else:
    #     break
# --- 5. 최종 결과 확인 ---
print("\n--- Inference Results (First 5) ---")
for res in results[:5]:
    print(f"Image: {res['img_path']}, Question: {res['question']}, Predicted Answer: {res['predicted_answer']}")
    
print('\n✅ Done.')

Answer to Index mapping created successfully.
Starting inference with choice scoring...


  0%|          | 1/680 [00:00<01:18,  8.68it/s]

100%|██████████| 680/680 [01:27<00:00,  7.81it/s]


--- Inference Results (First 5) ---
Image: ./input_images/TEST_000.jpg, Question: Which of the following foods is not present in the image?, Predicted Answer: Pizza
Image: ./input_images/TEST_001.jpg, Question: What might be the purpose of the person's workout in the image?, Predicted Answer: Building muscle and strength
Image: ./input_images/TEST_002.jpg, Question: Where might the family be headed?, Predicted Answer: To a business meeting
Image: ./input_images/TEST_003.jpg, Question: Where is the woman likely to be based on the background scenery?, Predicted Answer: In a forest
Image: ./input_images/TEST_004.jpg, Question: What is the woman in the picture doing?, Predicted Answer: Watching TV

✅ Done.


In [17]:
submission_df = pd.DataFrame(results)

# submission_df 에 ID 추가
submission_df['ID'] = submission_df['img_path'].apply(lambda x: re.sub(r'\.jpg$', '', os.path.basename(x)))

submission_df.head(5)

,img_path,question,predicted_answer,ID
0,./input_images/TEST_000.jpg,Which of the following foods is not present in...,Pizza,TEST_000
1,./input_images/TEST_001.jpg,What might be the purpose of the person's work...,Building muscle and strength,TEST_001
2,./input_images/TEST_002.jpg,Where might the family be headed?,To a business meeting,TEST_002
3,./input_images/TEST_003.jpg,Where is the woman likely to be based on the b...,In a forest,TEST_003
4,./input_images/TEST_004.jpg,What is the woman in the picture doing?,Watching TV,TEST_004


In [20]:
# 결과 A,B,C,D 중 하나로 만들기
merged_df = pd.merge(submission_df, test, on='ID')

# 2. 답변 내용을 A, B, C, D 문자로 변환하는 함수를 정의합니다.
def convert_text_to_choice(row):
    """
    한 행(row)을 입력받아 'predicted_answer'의 내용과
    'A', 'B', 'C', 'D' 열의 내용을 비교하여 일치하는 문자를 반환합니다.
    """
    predicted_text = row['predicted_answer']
    if predicted_text == row['A']:
        return 'A'
    elif predicted_text == row['B']:
        return 'B'
    elif predicted_text == row['C']:
        return 'C'
    elif predicted_text == row['D']:
        return 'D'
    else:
        # 일치하는 보기가 없을 경우, 기본값으로 'A'를 반환하고 경고 메시지를 출력합니다.
        # print(f"Warning: ID {row['ID']}의 예측 답변 '{predicted_text}'가 보기와 일치하지 않습니다.")
        return 'A' # 또는 None

# 3. apply 함수를 사용하여 모든 행에 대해 변환 함수를 적용합니다.
merged_df['answer'] = merged_df.apply(convert_text_to_choice, axis=1)

# 4. 제출 형식에 맞게 'ID'와 'answer'(A,B,C,D) 컬럼만 선택합니다.
submission_df = merged_df[['ID', 'answer']]


# --- 최종 결과 확인 ---
print("--- 변환 후 최종 제출용 데이터프레임 ---")
print(submission_df)


KeyError: 'predicted_answer'

In [21]:
submission_df.to_csv('finetuning_2.csv', index=False)


In [13]:
# 예측할 이미지와 질문
image_path = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images/TEST_000.jpg"
question = "Which of the following foods is not present in the image?,Pizza,Blueberries,Salmon,Avocado"

row = {
    'A': 'Pizza',
    'B': 'Blueberries',
    'C': 'Salmon',
    'D': 'Avocado'
}
choices = [row[c] for c in ['A', 'B', 'C', 'D']]

prompt = (
    "You are a helpful AI that answers multiple-choice questions based on the given image.\n"
    "Select the best answer from A, B, C, or D.\n\n"
    f"Question: Which of the following foods is not present in the image?\n"
    + "\n".join([f"{chr(65+i)}. {choice}" for i, choice in enumerate(choices)]) +
    "\nAnswer:"
)

# ▼▼▼▼▼ 텍스트(질문) 전처리 수정 ▼▼▼▼▼
# tokenizer.encode 대신 tokenizer.encode_plus 또는 그냥 tokenizer()를 사용합니다.
encoding = tokenizer(
    prompt,
    padding='max_length',       # 문장을 최대 길이에 맞춰 패딩
    truncation=True,            # 최대 길이보다 길면 자르기
    max_length=40,              # 모델 생성 시 사용한 max_text_len과 동일하게 설정
    return_tensors='pt'         # PyTorch 텐서로 반환
)
question_tokens = encoding['input_ids'].to(device)
# attention_mask를 생성하여 padding_mask로 사용합니다.
padding_mask = encoding['attention_mask'].to(device)
# ▲▲▲▲▲ 텍스트(질문) 전처리 수정 ▲▲▲▲▲

# --- 4. 추론 실행 ---
with torch.no_grad(): # 그래디언트 계산 비활성화
    # ▼▼▼▼▼ 모델 호출 부분 수정 ▼▼▼▼▼
    # 생성한 padding_mask를 세 번째 인자로 전달합니다.
    output = model(image=image_tensor, question=question_tokens, padding_mask=padding_mask)
    # ▲▲▲▲▲ 모델 호출 부분 수정 ▲▲▲▲▲

In [35]:
count = 0

import torch
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
import pandas as pd
import os
import re

# --- 1. 기본 설정 및 데이터 로드 ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_csv_path = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/test.csv'
image_dir = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/'
test = pd.read_csv(test_csv_path)
results = []

# --- 2. 이미지 전처리(Transform) 정의 ---
transform = transforms.Compose([
    transforms.Resize((768, 768)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

# --- 3. 답변 어휘 사전 및 'yes' 인덱스 준비 (★★ 핵심 준비 단계 ★★) ---
try:
    answer_vocab = list(answer_map.values())
    answer_to_idx = {answer: i for i, answer in enumerate(answer_vocab)}
    # 'yes'의 인덱스를 미리 찾아둡니다.
    if 'yes' in answer_to_idx:
        yes_idx = answer_to_idx['yes']
        print(f"Index for 'yes' found: {yes_idx}")
    else:
        raise ValueError("'yes' not found in answer vocabulary!")
except (NameError, ValueError) as e:
    print(f"Error preparing answer vocabulary: {e}")
    # 'yes'가 없으면 이 로직은 작동하지 않으므로 스크립트를 중단하는 것이 좋습니다.
    yes_idx = -1


# --- 4. 루프를 돌며 추론 실행 ---
print("Starting inference with Yes/No scoring...")
if yes_idx != -1:
    for _, row in tqdm(test.iterrows(), total=len(test)):
        # 4-1. 이미지 불러오기 및 전처리
        image_path = os.path.join(image_dir, row['img_path'].lstrip('./'))
        if not os.path.exists(image_path):
            results.append({'img_path': row['img_path'], 'predicted_answer': '?'})
            continue
        
        try:
            image = Image.open(image_path).convert("RGB")
            image_tensor = transform(image).unsqueeze(0).to(device)
        except Exception as e:
            results.append({'img_path': row['img_path'], 'predicted_answer': '?'})
            continue

        # 4-2. 각 보기에 대해 'yes' 점수를 계산 (★★ 핵심 로직 ★★)
        choices = [row[c] for c in ['A', 'B', 'C', 'D']]
        best_yes_score = -float('inf')
        predicted_answer = "N/A" # 만약 모든 보기의 yes 점수가 낮으면 N/A

        # 4개의 보기를 하나씩 모델에 넣어봅니다.
        for choice in choices:
            # 질문과 보기를 결합하여 새로운 프롬프트를 생성합니다.
            # 이 프롬프트 형식은 실험을 통해 더 좋은 형식을 찾을 수 있습니다.
            prompt = f"{row['Question']} {choice}"
            
            encoding = tokenizer(
                prompt, padding='max_length', truncation=True,
                max_length=40, return_tensors='pt'
            )
            question_tokens = encoding['input_ids'].to(device)
            padding_mask = encoding['attention_mask'].to(device)

            with torch.no_grad():
                output_logits = model(
                    image=image_tensor, 
                    question=question_tokens, 
                    padding_mask=padding_mask
                )
            
            # 전체 점수(logits)에서 'yes'에 해당하는 점수를 가져옵니다.
            yes_score = output_logits[0, yes_idx].item()
            
            # 현재 'yes' 점수가 최고 점수보다 높으면, 최고 점수와 정답을 업데이트
            if yes_score > best_yes_score:
                best_yes_score = yes_score
                predicted_answer = choice
        
        # 4-3. 최종 결과 저장
        results.append({
            'img_path': row['img_path'],
            'question': row['Question'],
            'predicted_answer': predicted_answer
        })
        if count < 5:
            print(f"Image: {row['img_path']}, Question: {row['Question']}, Predicted Answer: {predicted_answer}")
            count += 1
        else:
            break

# --- 5. 최종 결과 확인 ---
print("\n--- Inference Results (First 5) ---")
for res in results[:5]:
    print(f"Image: {res['img_path']}, Question: {res['question']}, Predicted Answer: {res['predicted_answer']}")
    
print('\n✅ Done.')

Index for 'yes' found: 0
Starting inference with Yes/No scoring...


  0%|          | 1/680 [00:00<05:01,  2.25it/s]

Image: ./input_images/TEST_000.jpg, Question: Which of the following foods is not present in the image?, Predicted Answer: Avocado


  0%|          | 2/680 [00:00<05:07,  2.21it/s]

Image: ./input_images/TEST_001.jpg, Question: What might be the purpose of the person's workout in the image?, Predicted Answer: Practicing for a marathon


...

Image: ./input_images/TEST_003.jpg, Question: Where is the woman likely to be based on the background scenery?, Predicted Answer: In a city park


  1%|          | 5/680 [00:02<05:01,  2.24it/s]

Image: ./input_images/TEST_004.jpg, Question: What is the woman in the picture doing?, Predicted Answer: Reading a book


  1%|          | 5/680 [00:02<06:07,  1.84it/s]


--- Inference Results (First 5) ---
Image: ./input_images/TEST_000.jpg, Question: Which of the following foods is not present in the image?, Predicted Answer: Avocado
Image: ./input_images/TEST_001.jpg, Question: What might be the purpose of the person's workout in the image?, Predicted Answer: Practicing for a marathon
Image: ./input_images/TEST_002.jpg, Question: Where might the family be headed?, Predicted Answer: To a school
Image: ./input_images/TEST_003.jpg, Question: Where is the woman likely to be based on the background scenery?, Predicted Answer: In a city park
Image: ./input_images/TEST_004.jpg, Question: What is the woman in the picture doing?, Predicted Answer: Reading a book

✅ Done.


In [14]:
# --- 5. 결과 후처리 ---
predicted_answer_index = output.argmax(-1).item()
predicted_answer = answer_map[str(predicted_answer_index)]

print(f"Question: {question}")
print(f"Predicted Answer: {predicted_answer}")

Question: Which of the following foods is not present in the image?,Pizza,Blueberries,Salmon,Avocado
Predicted Answer: tulips
